# 06. Experiment E: cross-encoder rerank

Part of Step 6's seven-notebook series: [01](01_dataset_and_chunking.ipynb) dataset/chunking, [02](02_experiment_a_chunking_strategy.ipynb) Experiment A, [03](03_experiment_b_search_mode.ipynb) Experiment B, [04](04_experiment_c_embedding_model.ipynb) Experiment C, [05](05_experiment_d_weights_and_rrf.ipynb) Experiment D, [06](06_experiment_e_rerank.ipynb) Experiment E, [07](07_calibration_and_analysis.ipynb) calibration/PCA/latency/failure analysis/conclusions. Each runs standalone against its own fresh ingest (via `_setup.py`); none depends on another's kernel state.

`cross-encoder/ms-marco-MiniLM-L6-v2` scoring the top 20 fused candidates,
replacing the calibrated cosine similarity with a sigmoid-squashed
cross-encoder logit, against the same pipeline with rerank off.


In [1]:
import _setup as s
import pandas as pd

W1031 11:14:17.840000 88356 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
rerank_results = {}
for rerank in [False, True]:
    rerank_results[rerank] = s.run_eval(
        model="bge-small", chunking="section", mode="hybrid", rerank=rerank, warm_runs=5
    )

rerank_df = pd.DataFrame(
    {
        ("rerank" if r else "no rerank"): {
            **res["retrieval_accuracy"]["macro_avg"],
            "p50_total_ms": res["latency"]["query"]["macro_avg"]["total"]["p50_ms"],
        }
        for r, res in rerank_results.items()
    }
).T
rerank_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,precision_at_5,precision_at_10,recall_at_10,mrr,ndcg_at_10,hit_at_1,p50_total_ms
no rerank,0.966667,0.8,0.712963,1.0,0.944259,1.0,48.013479
rerank,1.000000,0.8,0.712963,1.0,0.935956,1.0,166.904410


In [3]:
for name, row in rerank_df.iterrows():
    print(f"{name:<10} nDCG@10={row['ndcg_at_10']:.3f}  p50={row['p50_total_ms']:.1f}ms")

no rerank  nDCG@10=0.944  p50=48.0ms
rerank     nDCG@10=0.936  p50=166.9ms


**Finding:** after Step 10's R1 fix, reranking here gives a small nDCG@10 *loss* (0.944 -> 0.936) at roughly 3.5x the query latency, rather than the small gain measured before that fix. The earlier "small gain" reading was measured against the same R1-inflated baseline every other Step-6 number was; on the corrected pipeline, the cross-encoder's own ranking of the top-20 pool isn't better than the already-strong hybrid RRF order here. `--rerank` stays off by default (see `PLAN.md`'s design-decisions table); R19 (Step 11) revisits rerank scope and scale.
